# Volume Access Grants for App SPN

Grants `READ_VOLUME` and `WRITE_VOLUME` on a single UC Volume to the lakeLoom App's
auto-provisioned service principal. Required for the App to write uploaded files
(audio, screenshots, photos, documents) to UC Volumes via the WorkspaceClient SDK.

This notebook is invoked by the `configure_app_spn` job as a **forEach task**,
iterating over the list of managed volumes. Each iteration receives a
`volume_name` parameter identifying which volume to grant access on.

Parameters (passed from job definition):
- `catalog_use` — Unity Catalog catalog name (e.g., `hls_fde_dev`)
- `schema_use` — Unity Catalog schema name (e.g., `dev_matthew_giglia_lakeloom`)
- `spn_application_id` — App SPN application_id (UUID)
- `volume_name` — from forEach input list (e.g. `session_audio`, `screenshots`, `documents`)

In [0]:
DECLARE OR REPLACE VARIABLE catalog_use STRING;
DECLARE OR REPLACE VARIABLE schema_use STRING;

SET VARIABLE catalog_use = :catalog_use;
SET VARIABLE schema_use = :schema_use;

USE IDENTIFIER(catalog_use || '.' || schema_use);
SELECT current_catalog() AS active_catalog, current_schema() AS active_schema;

In [0]:
SET QUERY_TAGS['project'] = 'lakeLoom';
SET QUERY_TAGS['component'] = 'configure_app_spn';
SET QUERY_TAGS['pipeline'] = 'lakeloom_ai';
EXECUTE IMMEDIATE "SET QUERY_TAGS['catalog'] = '" || catalog_use || "';";
EXECUTE IMMEDIATE "SET QUERY_TAGS['schema'] = '" || schema_use || "';";

In [0]:
DECLARE OR REPLACE VARIABLE spn_application_id STRING;
SET VARIABLE spn_application_id = :spn_application_id;

DECLARE OR REPLACE VARIABLE volume_name STRING;
SET VARIABLE volume_name = :volume_name;

SELECT
  spn_application_id AS granting_to,
  volume_name AS target_volume,
  catalog_use || '.' || schema_use || '.' || volume_name AS fully_qualified_volume;

In [0]:
DECLARE OR REPLACE VARIABLE read_vol_grnt_stmnt STRING DEFAULT
  'GRANT READ VOLUME ON VOLUME ' || catalog_use || '.' || schema_use || '.' || volume_name || ' TO `' || spn_application_id || '`;';

SELECT read_vol_grnt_stmnt;

In [0]:
EXECUTE IMMEDIATE read_vol_grnt_stmnt;

In [0]:
DECLARE OR REPLACE VARIABLE write_vol_grnt_stmnt STRING DEFAULT
  'GRANT WRITE VOLUME ON VOLUME ' || catalog_use || '.' || schema_use || '.' || volume_name || ' TO `' || spn_application_id || '`;';

SELECT write_vol_grnt_stmnt;

In [0]:
EXECUTE IMMEDIATE write_vol_grnt_stmnt;

In [0]:
EXECUTE IMMEDIATE 'SHOW GRANTS ON VOLUME ' || catalog_use || '.' || schema_use || '.' || volume_name;

In [0]:
SELECT
  'volume_grants_applied' AS check_name,
  'read_write_granted' AS status,
  catalog_use || '.' || schema_use || '.' || volume_name AS target_volume,
  spn_application_id AS granted_to,
  current_timestamp() AS completed_at;